## Pre - Processing 

### Patient-Based Partitioning 
- The BEED dataset from the UCI repository is in a tabular format, so every 200 lines represents one patient. Patient-Based Partitioning will be used to ensure that a patient doesn't end up in more than one place. 

### Channel-Wise Z-Score Normalization
- EEG amplitude levels natively vary across channels, so unscaled features can distort weight updates and slow down training. Standardization centers the data $(X_{\text{norm}} = \frac{X - \mu}{\sigma})$ so every channel contributes equally to gradient descent.


In [ ]:
# Patient ID Grouping

import pandas as pd 
#Load data into a DataFrame
df = pd.read_csv("BEED_Data.csv")

# Create a 'Patient_ID' column
df['Patient_ID'] = df.index // 200 + 1

# Create Groups 
groups = df.groupby('Patient_ID')


In [ ]:
from sklearn.model_selection import GroupShuffleSplit

# Isolate the Test Set (15% Test, 85% Train/Validation)
gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
train_val_idx, test_idx = next(gss.split(df, groups=df['Patient_ID']))

train_val_df = df.iloc[train_val_idx]
test_val_df = df.iloc[test_idx]

# Divide up the Train/Validation Set (70% Train, 15% Validation)

gss = GroupShuffleSplit(n_splits=1, test_size=0.1765, random_state=42)
train_idx, val_idx = next(gss.split(train_val_df, groups=train_val_df['Patient_ID']))

train_df = train_val_df.iloc[train_idx]
val_df = train_val_df.iloc[val_idx]

# Calculate intersections (to ensure no overlap between sets)
train_patients = set(train_df['Patient_ID'])
val_patients = set(val_df['Patient_ID'])
test_patients = set(test_val_df['Patient_ID'])

# Verify that there are no overlapping patients between the sets
print("Leakage Results:")
print(f"Overlap between Train and Validation sets: {train_patients.intersection(val_patients)}")
print(f"Overlap between Train and Test sets: {train_patients.intersection(test_patients)}")
print(f"Overlap between Validation and Test sets: {val_patients.intersection(test_patients)}")

# Trigger error if there are overlapping patients
assert (
    len(train_patients.intersection(val_patients)) == 0
    and len(train_patients.intersection(test_patients)) == 0
    and len(val_patients.intersection(test_patients)) == 0
), "Overlap between Train, Validation, and Test sets"

print("No overlapping patients between Train, Validation, and Test sets. Data splitting successful.")


In [14]:
# Z-Score Normalization

from sklearn.preprocessing import StandardScaler
import torch 
from torch.utils.data import DataLoader, TensorDataset
scaler = StandardScaler()

# Define target column and feature columns
target_column = 'y'
feature_columns = [col for col in df.columns if col != target_column and col != 'Patient_ID']

# Extract feature arrays and target arrays
X_train= train_df[feature_columns].values
y_train = train_df[target_column].values

X_val = val_df[feature_columns].values
y_val = val_df[target_column].values

X_test = test_val_df[feature_columns].values
y_test = test_val_df[target_column].values

# Z-score normalization for training data
X_train_scaled = scaler.fit_transform(X_train)

# Z-score normalization for validation data
X_val_scaled = scaler.transform(X_val)

# Z-score normalization for test data
X_test_scaled = scaler.transform(X_test)

# Convert the scaled data to PyTorch tensors
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)

X_val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32)

X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=64, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_tensor, y_val_tensor), batch_size=64, shuffle=False)
test_loader = DataLoader(TensorDataset(X_test_tensor, y_test_tensor), batch_size=64, shuffle=False)

print("Data loading complete. Train, Validation, and Test DataLoaders are ready.")

Data loading complete. Train, Validation, and Test DataLoaders are ready.
